# ⭐ Day 54: Random Forest  
## The Power of Ensembles | Complete Tutorial with Examples & Exercises

**Day 54 of 369-day Python & AI Learning Path** 🚀


## Welcome to Day 54! 🌲🌲🌲

Yesterday we mastered the Decision Tree — a powerful yet simple algorithm. But as we discovered, single trees often overfit and can be unstable. Today, we unlock the next level: **Random Forest**, one of the most widely used and effective machine learning algorithms in existence.

Random Forest operates on a beautiful principle: *the wisdom of the crowd*. By combining hundreds of diverse Decision Trees, each trained on slightly different data and features, we create an ensemble that is far more accurate and robust than any single tree could ever be. It's like asking a forest of experts instead of just one — the collective decision is almost always better.

This algorithm dominates Kaggle competitions for tabular data, powers countless production systems at major tech companies, and serves as the foundation for even more advanced methods like Gradient Boosting. Whether you're predicting customer churn, diagnosing diseases, or forecasting sales, Random Forest should be in your toolkit.

By the end of today, you'll understand exactly how bagging and random feature selection work, how to tune Random Forest hyperparameters like a pro, and why this algorithm strikes the perfect balance between accuracy and interpretability. Let's grow our forest! 🌳


## 📋 Table of Contents

1. [Introduction to Ensemble Learning and Bagging](#1-introduction-to-ensemble-learning-and-bagging)
2. [Why Random Forest Works So Well](#2-why-random-forest-works-so-well)
3. [Key Concepts: Bagging, Random Features, OOB](#3-key-concepts)
4. [Building Random Forest from Scratch](#4-building-from-scratch)
5. [Scikit-Learn Implementation](#5-scikit-learn-implementation)
6. [Hyperparameters Deep Dive](#6-hyperparameters-deep-dive)
7. [Titanic Dataset Application](#7-titanic-dataset)
8. [Model Evaluation and Comparison](#8-model-evaluation)
9. [Feature Importance Analysis](#9-feature-importance)
10. [Overfitting Control](#10-overfitting-control)
11. [Pros, Cons, and When to Use](#11-pros-cons-when-to-use)
12. [Hands-On Exercises](#hands-on-exercises)
13. [Solutions](#solutions)


## 1. Introduction to Ensemble Learning and Bagging 🎯

### What is Ensemble Learning?

Ensemble learning combines multiple machine learning models to produce better predictive performance than any single model could achieve alone. The core idea is simple: *a group of weak learners can form a strong learner*.

### Types of Ensembles:
- **Bagging** (Bootstrap Aggregating): Train models in parallel on random subsets
- **Boosting**: Train models sequentially, each correcting errors of previous
- **Stacking**: Combine different types of models using a meta-learner

### Bootstrap Aggregating (Bagging):
1. Create multiple bootstrap samples (random sampling with replacement)
2. Train a model on each sample
3. Aggregate predictions (majority vote for classification, average for regression)

> 💡 **Why Bagging Works**: By averaging many noisy but unbiased models, we reduce variance without increasing bias!


## 2. Why Random Forest Works So Well 🧠

Random Forest combines two powerful ideas:

### 1. Wisdom of the Crowd
Individual trees may make mistakes, but they make *different* mistakes. When we aggregate hundreds of trees, errors cancel out and correct predictions reinforce each other.

### 2. Decorrelation Through Randomness
If trees are too similar, bagging won't help much. Random Forest introduces additional randomness by:
- **Bootstrap sampling**: Each tree sees different training data
- **Feature randomness**: At each split, only a random subset of features is considered

This ensures trees are diverse (low correlation) while still being accurate.

### The Magic Formula:
$$\text{Forest Error} \leq \bar{\rho} \times \text{(Average Tree Error)}$$

Where $\bar{\rho}$ is the correlation between trees. Lower correlation = better ensemble!


## 3. Key Concepts 📚

### Bootstrap Aggregating (Bagging)
- Sample $n$ instances with replacement from training set
- About 63.2% of unique samples appear in each bootstrap sample
- Remaining 36.8% are "out-of-bag" (OOB) samples

### Random Feature Selection
- At each split, consider only $m$ features randomly selected from total $p$ features
- Typically $m = \sqrt{p}$ for classification, $m = p/3$ for regression
- Forces trees to explore different paths

### Out-of-Bag (OOB) Score
- Natural validation set for each tree
- No need for separate cross-validation
- Unbiased estimate of test error

> 💡 **Practical Tip**: Always enable `oob_score=True` when training to get a free validation metric!


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")
print("📦 Ready to build Random Forests!")


## 4. Building a Random Forest from Scratch (Simplified) 🛠️

Let's implement a basic Random Forest to understand the mechanics.


In [ ]:
class SimpleRandomForest:
    """
    Simplified Random Forest for educational purposes.
    """
    def __init__(self, n_estimators=10, max_depth=5, max_features='sqrt', random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.feature_indices = []
        
    def _bootstrap_sample(self, X, y):
        """Create bootstrap sample"""
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        return X[indices], y[indices]
    
    def _get_feature_subset(self, n_features):
        """Select random subset of features"""
        if self.max_features == 'sqrt':
            k = int(np.sqrt(n_features))
        elif self.max_features == 'log2':
            k = int(np.log2(n_features))
        else:
            k = n_features
        return np.random.choice(n_features, size=k, replace=False)
    
    def fit(self, X, y):
        """Train the forest"""
        np.random.seed(self.random_state)
        n_features = X.shape[1]
        
        for i in range(self.n_estimators):
            # Bootstrap sample
            X_boot, y_boot = self._bootstrap_sample(X, y)
            
            # Select feature subset for this tree
            feat_idx = self._get_feature_subset(n_features)
            self.feature_indices.append(feat_idx)
            
            # Train tree
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=self.random_state)
            tree.fit(X_boot[:, feat_idx], y_boot)
            self.trees.append(tree)
            
        return self
    
    def predict(self, X):
        """Make predictions by majority voting"""
        predictions = np.zeros((X.shape[0], self.n_estimators))
        
        for i, tree in enumerate(self.trees):
            predictions[:, i] = tree.predict(X[:, self.feature_indices[i]])
        
        # Majority vote
        return np.array([np.bincount(row.astype(int)).argmax() for row in predictions])

# Test on synthetic data
from sklearn.datasets import make_classification
X_rf, y_rf = make_classification(n_samples=500, n_features=6, n_informative=4, random_state=42)
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)

custom_rf = SimpleRandomForest(n_estimators=20, max_depth=5, random_state=42)
custom_rf.fit(X_train_rf, y_train_rf)
custom_pred = custom_rf.predict(X_test_rf)

print("🌲 Custom Random Forest Results:")
print(f"✅ Accuracy: {accuracy_score(y_test_rf, custom_pred):.4f}")
print(f"🌳 Number of trees: {len(custom_rf.trees)}")


## 5. Using Scikit-Learn RandomForestClassifier 🚀

Now let's use the production-ready implementation.


In [ ]:
# Initialize Random Forest with key parameters
rf_classifier = RandomForestClassifier(
    n_estimators=100,        # Number of trees
    max_depth=10,            # Maximum depth of trees
    min_samples_split=5,     # Minimum samples to split
    min_samples_leaf=2,      # Minimum samples in leaf
    max_features='sqrt',     # Features to consider at each split
    bootstrap=True,          # Use bootstrap samples
    oob_score=True,          # Enable OOB scoring
    random_state=42,
    n_jobs=-1                # Use all CPU cores
)

rf_classifier.fit(X_train_rf, y_train_rf)
rf_pred = rf_classifier.predict(X_test_rf)

print("🚀 Scikit-Learn Random Forest Results:")
print(f"✅ Test Accuracy: {accuracy_score(y_test_rf, rf_pred):.4f}")
print(f"📊 OOB Score: {rf_classifier.oob_score_:.4f}")
print(f"🌲 Number of estimators: {rf_classifier.n_estimators}")
print(f"📏 Average tree depth: {np.mean([tree.tree_.max_depth for tree in rf_classifier.estimators_]):.1f}")


## 6. Hyperparameters Deep Dive ⚙️

### Key Parameters:

**n_estimators**: Number of trees in the forest
- More trees = better performance but slower training
- Usually 100-500 is sufficient; diminishing returns after that

**max_depth**: Maximum depth of each tree
- Deeper trees = more complex model
- None means nodes expand until pure or min_samples_split reached

**max_features**: Features to consider at each split
- 'sqrt': $\sqrt{n}$ features (default for classification)
- 'log2': $\log_2(n)$ features
- None: All features (no randomness, reduces to bagging)

**min_samples_split**: Minimum samples required to split
- Higher values = more constrained trees, less overfitting

**min_samples_leaf**: Minimum samples in leaf node
- Higher values = smoother decision boundaries

**bootstrap**: Whether to use bootstrap samples
- False = use whole dataset (less diversity)


In [ ]:
# Effect of n_estimators (Learning Curve)
estimator_range = [1, 5, 10, 25, 50, 75, 100, 150, 200]
train_scores = []
test_scores = []
oob_scores = []

for n in estimator_range:
    rf = RandomForestClassifier(
        n_estimators=n, 
        max_depth=5, 
        oob_score=True,
        random_state=42
    )
    rf.fit(X_train_rf, y_train_rf)
    train_scores.append(rf.score(X_train_rf, y_train_rf))
    test_scores.append(rf.score(X_test_rf, y_test_rf))
    oob_scores.append(rf.oob_score_)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(estimator_range, train_scores, 'o-', label='Training Score', linewidth=2, color='#2ECC71')
ax.plot(estimator_range, test_scores, 's-', label='Test Score', linewidth=2, color='#E74C3C')
ax.plot(estimator_range, oob_scores, '^-', label='OOB Score', linewidth=2, color='#3498DB')
ax.set_xlabel('Number of Estimators (Trees)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Random Forest Learning Curve\n(Effect of n_estimators)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 1.05)

plt.tight_layout()
plt.show()

print("📊 Observation: Performance stabilizes around 50-100 trees")
print("📊 OOB score closely tracks test performance - great for validation!")


In [ ]:
# Grid Search for hyperparameter tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, oob_score=True),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_rf, y_train_rf)

print("⚙️ Best Hyperparameters:")
print(grid_search.best_params_)
print(f"🎯 Best CV Score: {grid_search.best_score_:.4f}")
print(f"🎯 Test Score: {grid_search.score(X_test_rf, y_test_rf):.4f}")


## 7. Applying Random Forest to Titanic Dataset 🚢

Let's apply our knowledge to the Titanic survival prediction.


In [ ]:
# Load and preprocess Titanic
titanic = sns.load_dataset('titanic')

def preprocess_titanic(df):
    data = df.copy()
    features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
    data = data[features + ['survived']]
    
    # Fill missing values
    data['age'].fillna(data['age'].median(), inplace=True)
    data['embarked'].fillna(data['embarked'].mode()[0], inplace=True)
    data['fare'].fillna(data['fare'].median(), inplace=True)
    
    # Encode
    le = LabelEncoder()
    data['sex'] = le.fit_transform(data['sex'])
    data['embarked'] = le.fit_transform(data['embarked'])
    
    return data

titanic_clean = preprocess_titanic(titanic)
X_titanic = titanic_clean.drop('survived', axis=1)
y_titanic = titanic_clean['survived']
feature_names = X_titanic.columns.tolist()

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42, stratify=y_titanic)

print("🚢 Titanic Data Preprocessed!")
print(f"Training set: {X_train_t.shape}")
print(f"Test set: {X_test_t.shape}")


In [ ]:
# Train Random Forest on Titanic
rf_titanic = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

rf_titanic.fit(X_train_t, y_train_t)
rf_pred_t = rf_titanic.predict(X_test_t)

print("🚀 Random Forest Titanic Results:")
print(f"✅ Test Accuracy: {accuracy_score(y_test_t, rf_pred_t):.4f}")
print(f"📊 OOB Score: {rf_titanic.oob_score_:.4f}")
print(f"📈 Training Score: {rf_titanic.score(X_train_t, y_train_t):.4f}")
print("\n📋 Classification Report:")
print(classification_report(y_test_t, rf_pred_t, target_names=['Did Not Survive', 'Survived']))


## 8. Model Evaluation and Comparison 📊

Let's compare Random Forest with a single Decision Tree.


In [ ]:
# Train single Decision Tree for comparison
dt_titanic = DecisionTreeClassifier(max_depth=8, random_state=42)
dt_titanic.fit(X_train_t, y_train_t)
dt_pred_t = dt_titanic.predict(X_test_t)

# Cross-validation comparison
rf_cv = cross_val_score(rf_titanic, X_titanic, y_titanic, cv=5)
dt_cv = cross_val_score(dt_titanic, X_titanic, y_titanic, cv=5)

print("⚔️ Random Forest vs Decision Tree:\n")
print(f"Decision Tree:")
print(f"  Test Accuracy: {accuracy_score(y_test_t, dt_pred_t):.4f}")
print(f"  CV Mean: {dt_cv.mean():.4f} (+/- {dt_cv.std()*2:.4f})")
print(f"  Training Score: {dt_titanic.score(X_train_t, y_train_t):.4f}")

print(f"\nRandom Forest:")
print(f"  Test Accuracy: {accuracy_score(y_test_t, rf_pred_t):.4f}")
print(f"  CV Mean: {rf_cv.mean():.4f} (+/- {rf_cv.std()*2:.4f})")
print(f"  Training Score: {rf_titanic.score(X_train_t, y_train_t):.4f}")

print(f"\n🎯 Improvement: {accuracy_score(y_test_t, rf_pred_t) - accuracy_score(y_test_t, dt_pred_t):.4f}")
print("💡 Random Forest reduces variance while maintaining low bias!")


In [ ]:
# Visual comparison
models = ['Decision Tree', 'Random Forest']
test_scores = [accuracy_score(y_test_t, dt_pred_t), accuracy_score(y_test_t, rf_pred_t)]
cv_means = [dt_cv.mean(), rf_cv.mean()]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, test_scores, width, label='Test Accuracy', color='#3498DB')
bars2 = ax.bar(x + width/2, cv_means, width, label='CV Mean', color='#E74C3C')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Decision Tree vs Random Forest Performance', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0.7, 1.0)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# ROC Curve Comparison
rf_proba = rf_titanic.predict_proba(X_test_t)[:, 1]
dt_proba = dt_titanic.predict_proba(X_test_t)[:, 1]

fpr_rf, tpr_rf, _ = roc_curve(y_test_t, rf_proba)
fpr_dt, tpr_dt, _ = roc_curve(y_test_t, dt_proba)
roc_auc_rf = auc(fpr_rf, tpr_rf)
roc_auc_dt = auc(fpr_dt, tpr_dt)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(fpr_rf, tpr_rf, color='#2ECC71', lw=2, label=f'Random Forest (AUC = {roc_auc_rf:.3f})')
ax.plot(fpr_dt, tpr_dt, color='#E74C3C', lw=2, label=f'Decision Tree (AUC = {roc_auc_dt:.3f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', alpha=0.5)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Feature Importance Analysis 🌟

Random Forest provides reliable feature importance based on mean decrease in impurity.


In [ ]:
# Feature importance from Random Forest
importances = rf_titanic.feature_importances_
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(feature_names)))

bars = ax.bar(range(len(importances)), importances[indices], color=colors)
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([feature_names[i] for i in indices], rotation=45, ha='right')
ax.set_ylabel('Importance', fontsize=12)
ax.set_title('🌟 Random Forest Feature Importance\n(Titanic Survival)', fontsize=14, fontweight='bold')

for i, bar in enumerate(bars):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("📊 Feature Importance Ranking:")
for i in indices:
    print(f"{i+1}. {feature_names[i]}: {importances[i]:.4f}")


In [ ]:
# Compare feature importance: Single Tree vs Random Forest
dt_importances = dt_titanic.feature_importances_

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

x_pos = np.arange(len(feature_names))

# Random Forest
ax1.bar(x_pos, importances[indices], color='#2ECC71', alpha=0.8)
ax1.set_xticks(x_pos)
ax1.set_xticklabels([feature_names[i] for i in indices], rotation=45, ha='right')
ax1.set_title('Random Forest\nFeature Importance', fontsize=12, fontweight='bold')
ax1.set_ylabel('Importance')

# Decision Tree
dt_indices = np.argsort(dt_importances)[::-1]
ax2.bar(x_pos, dt_importances[dt_indices], color='#E74C3C', alpha=0.8)
ax2.set_xticks(x_pos)
ax2.set_xticklabels([feature_names[i] for i in dt_indices], rotation=45, ha='right')
ax2.set_title('Single Decision Tree\nFeature Importance', fontsize=12, fontweight='bold')
ax2.set_ylabel('Importance')

plt.tight_layout()
plt.show()

print("💡 Random Forest importance is more stable and reliable!")
print("💡 Single trees may overemphasize features due to early splits")


## 10. Overfitting Control and Best Practices 🛑

### Signs of Overfitting in Random Forest:
- Large gap between training and test scores
- OOB score much lower than training score
- Individual trees are very deep

### Best Practices:
1. **Use OOB score** for quick validation without separate CV
2. **Limit tree depth** (`max_depth`) to prevent deep trees
3. **Increase `min_samples_leaf`** for smoother predictions
4. **Cross-validate** final model performance
5. **Feature engineering** often helps more than tuning


In [ ]:
# Demonstrate overfitting control
depths = [2, 4, 6, 8, 10, 15, 20, None]
train_scores_d = []
test_scores_d = []
oob_scores_d = []

for depth in depths:
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=depth,
        oob_score=True,
        random_state=42
    )
    rf.fit(X_train_t, y_train_t)
    train_scores_d.append(rf.score(X_train_t, y_train_t))
    test_scores_d.append(rf.score(X_test_t, y_test_t))
    oob_scores_d.append(rf.oob_score_)

fig, ax = plt.subplots(figsize=(12, 7))
depth_labels = [str(d) for d in depths]
ax.plot(depth_labels, train_scores_d, 'o-', label='Training', linewidth=2, color='#2ECC71')
ax.plot(depth_labels, test_scores_d, 's-', label='Test', linewidth=2, color='#E74C3C')
ax.plot(depth_labels, oob_scores_d, '^-', label='OOB', linewidth=2, color='#3498DB')
ax.set_xlabel('Max Depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Random Forest: Effect of Tree Depth\n(Overfitting Analysis)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.7, 1.05)

plt.tight_layout()
plt.show()

print("📊 Observation: Unlike single trees, Random Forest is quite robust to overfitting!")
print("📊 However, very deep trees can still overfit slightly")


## 11. Pros, Cons, and When to Use Random Forest ⚖️

### ✅ Pros:
- **High accuracy**: Often among best out-of-box performers
- **Robust to overfitting**: Due to averaging many trees
- **Handles missing values**: Built-in support
- **Feature importance**: Automatic feature selection
- **No scaling needed**: Works with raw data
- **Parallelizable**: Fast training with `n_jobs=-1`
- **Handles non-linear relationships**: Naturally captures interactions

### ❌ Cons:
- **Less interpretable**: Black box compared to single tree
- **Memory intensive**: Must store hundreds of trees
- **Slow predictions**: Must run through all trees
- **Biased toward categorical features**: With many levels
- **Struggles with rare classes**: Imbalanced datasets need care

### 🎯 When to Use:
- Tabular data competitions (Kaggle)
- Baseline model for new projects
- Feature selection and importance analysis
- When you need robust, reliable predictions
- Production systems requiring stability

### 🚫 When NOT to Use:
- Real-time predictions (latency critical)
- Very high-dimensional sparse data (use linear models)
- When interpretability is legally required
- Extremely large datasets (consider Gradient Boosting)


## 🛠️ Hands-On Exercises

Time to practice! Complete these exercises to master Random Forest.

### Exercise 1: Effect of n_estimators
Train Random Forests with n_estimators = [10, 50, 100, 200, 500] on the Titanic dataset. Plot how accuracy and training time change.


In [ ]:
# Exercise 1: Your code here



### Exercise 2: Compare with Decision Tree
Train a single Decision Tree and Random Forest on the Wine dataset. Compare accuracy, training time, and stability using cross-validation.


In [ ]:
# Exercise 2: Your code here



### Exercise 3: Hyperparameter Tuning
Find the best combination of `max_depth` and `min_samples_split` for the Titanic dataset using GridSearchCV.


In [ ]:
# Exercise 3: Your code here



### Exercise 4: Feature Importance Deep Dive
Train a Random Forest on the Breast Cancer dataset (`sklearn.datasets.load_breast_cancer()`). Identify the top 5 most important features and visualize them.


In [ ]:
# Exercise 4: Your code here



### Exercise 5: OOB Score Validation
Demonstrate that OOB score is a good estimate of test accuracy by comparing OOB scores with 5-fold cross-validation scores across different datasets.


In [ ]:
# Exercise 5: Your code here



### Exercise 6: Random Forest for Regression
Use RandomForestRegressor on the Boston Housing dataset (or California Housing). Evaluate using RMSE and R² score. Compare with a single Decision Tree regressor.


In [ ]:
# Exercise 6: Your code here



### Exercise 7: Feature Selection
Use Random Forest feature importance to select the top 50% of features from the Titanic dataset. Retrain and compare performance with using all features.


In [ ]:
# Exercise 7: Your code here



### Exercise 8: Bootstrap Analysis
Calculate and visualize the distribution of OOB scores vs test scores across 20 different random seeds to show OOB reliability.


In [ ]:
# Exercise 8: Your code here



### Exercise 9: Robust Pipeline
Create a complete sklearn pipeline that includes preprocessing, Random Forest, and hyperparameter tuning. Test it on the Titanic dataset.


In [ ]:
# Exercise 9: Your code here



### Exercise 10: Real-world Challenge
Download the Heart Disease UCI dataset (or use `fetch_openml(name='heart-statlog')`). Build a Random Forest model to predict heart disease. Optimize for recall (sensitivity) since missing a heart disease case is costly.


In [ ]:
# Exercise 10: Your code here



## Solutions (Review After Attempting) ✅

Detailed solutions with explanations are below. Attempt the exercises first!


In [ ]:
# Solution 1: Effect of n_estimators
import time

n_estimators_list = [10, 50, 100, 200, 500]
accuracies = []
times = []

for n in n_estimators_list:
    start = time.time()
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train_t, y_train_t)
    elapsed = time.time() - start
    
    accuracies.append(rf.score(X_test_t, y_test_t))
    times.append(elapsed)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(n_estimators_list, accuracies, 'o-', linewidth=2, markersize=8, color='#3498DB')
ax1.set_xlabel('n_estimators')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('Accuracy vs Number of Trees')
ax1.grid(True, alpha=0.3)

ax2.plot(n_estimators_list, times, 's-', linewidth=2, markersize=8, color='#E74C3C')
ax2.set_xlabel('n_estimators')
ax2.set_ylabel('Training Time (seconds)')
ax2.set_title('Training Time vs Number of Trees')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Solution 1: Diminishing returns after ~100-200 trees")


In [ ]:
# Solution 2: Wine Dataset Comparison
from sklearn.datasets import load_wine
import time

wine = load_wine()
X_w, y_w = wine.data, wine.target
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_w, y_w, test_size=0.3, random_state=42)

# Decision Tree
start = time.time()
dt_w = DecisionTreeClassifier(random_state=42)
dt_cv_w = cross_val_score(dt_w, X_w, y_w, cv=5)
dt_time = time.time() - start

# Random Forest
start = time.time()
rf_w = RandomForestClassifier(n_estimators=100, random_state=42)
rf_cv_w = cross_val_score(rf_w, X_w, y_w, cv=5)
rf_time = time.time() - start

print("Solution 2 - Wine Dataset Results:")
print(f"Decision Tree: {dt_cv_w.mean():.4f} (+/- {dt_cv_w.std():.4f}) | Time: {dt_time:.3f}s")
print(f"Random Forest: {rf_cv_w.mean():.4f} (+/- {rf_cv_w.std():.4f}) | Time: {rf_time:.3f}s")
print(f"Improvement: {rf_cv_w.mean() - dt_cv_w.mean():.4f}")


In [ ]:
# Solution 3: Grid Search
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20]
}

grid = GridSearchCV(
    RandomForestClassifier(n_estimators=100, random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train_t, y_train_t)

print("Solution 3 - Best Parameters:")
print(grid.best_params_)
print(f"Best CV Score: {grid.best_score_:.4f}")
print(f"Test Score: {grid.score(X_test_t, y_test_t):.4f}")


In [ ]:
# Solution 4: Breast Cancer Feature Importance
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.3, random_state=42)

rf_cancer = RandomForestClassifier(n_estimators=100, random_state=42)
rf_cancer.fit(X_train_c, y_train_c)

# Top 5 features
importances_c = rf_cancer.feature_importances_
indices_c = np.argsort(importances_c)[::-1][:5]

plt.figure(figsize=(10, 6))
plt.bar(range(5), importances_c[indices_c], color='teal')
plt.xticks(range(5), [cancer.feature_names[i] for i in indices_c], rotation=45, ha='right')
plt.title('Solution 4: Top 5 Breast Cancer Features')
plt.tight_layout()
plt.show()

print("Solution 4 - Top 5 Features:")
for i in indices_c:
    print(f"  {cancer.feature_names[i]}: {importances_c[i]:.4f}")


In [ ]:
# Solution 5: OOB vs CV Comparison
from sklearn.datasets import load_iris, load_wine, load_breast_cancer

datasets = {
    'Iris': load_iris(),
    'Wine': load_wine(),
    'Breast Cancer': load_breast_cancer()
}

results = []

for name, data in datasets.items():
    X, y = data.data, data.target
    
    # OOB Score
    rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
    rf_oob.fit(X, y)
    oob = rf_oob.oob_score_
    
    # CV Score
    cv = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X, y, cv=5).mean()
    
    results.append({'Dataset': name, 'OOB': oob, 'CV': cv, 'Diff': abs(oob-cv)})

results_df = pd.DataFrame(results)
print("Solution 5 - OOB vs CV Comparison:")
print(results_df)
print(f"\nAverage difference: {results_df['Diff'].mean():.4f}")


In [ ]:
# Solution 6: Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X_h, y_h = housing.data, housing.target
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

# Decision Tree Regressor
dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train_h, y_train_h)
dt_pred_h = dt_reg.predict(X_test_h)

# Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_h, y_train_h)
rf_pred_h = rf_reg.predict(X_test_h)

print("Solution 6 - Regression Results:")
print(f"Decision Tree - RMSE: {np.sqrt(mean_squared_error(y_test_h, dt_pred_h)):.4f}, R²: {r2_score(y_test_h, dt_pred_h):.4f}")
print(f"Random Forest - RMSE: {np.sqrt(mean_squared_error(y_test_h, rf_pred_h)):.4f}, R²: {r2_score(y_test_h, rf_pred_h):.4f}")


In [ ]:
# Solution 7: Feature Selection
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X_train_t, y_train_t)
full_score = rf_full.score(X_test_t, y_test_t)

# Select top 50% features
importances_t = rf_full.feature_importances_
n_select = len(importances_t) // 2
top_features = np.argsort(importances_t)[::-1][:n_select]

X_train_selected = X_train_t.iloc[:, top_features]
X_test_selected = X_test_t.iloc[:, top_features]

rf_selected = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selected.fit(X_train_selected, y_train_t)
selected_score = rf_selected.score(X_test_selected, y_test_t)

print("Solution 7 - Feature Selection:")
print(f"All features ({len(importances_t)}): {full_score:.4f}")
print(f"Top 50% ({n_select}): {selected_score:.4f}")
print(f"Selected: {[feature_names[i] for i in top_features]}")


In [ ]:
# Solution 8: OOB Reliability Analysis
oob_scores_r = []
test_scores_r = []
seeds = range(20)

for seed in seeds:
    rf_r = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=seed)
    rf_r.fit(X_train_t, y_train_t)
    oob_scores_r.append(rf_r.oob_score_)
    test_scores_r.append(rf_r.score(X_test_t, y_test_t))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(seeds, oob_scores_r, 'o-', label='OOB', alpha=0.7)
plt.plot(seeds, test_scores_r, 's-', label='Test', alpha=0.7)
plt.xlabel('Random Seed')
plt.ylabel('Accuracy')
plt.title('OOB vs Test Scores Across Seeds')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(np.array(oob_scores_r) - np.array(test_scores_r), bins=10, edgecolor='black', alpha=0.7)
plt.xlabel('OOB - Test Difference')
plt.ylabel('Frequency')
plt.title('Distribution of Differences')
plt.axvline(0, color='red', linestyle='--')

plt.tight_layout()
plt.show()

print(f"Solution 8 - Mean difference: {np.mean(np.array(oob_scores_r) - np.array(test_scores_r)):.4f}")
print(f"Std difference: {np.std(np.array(oob_scores_r) - np.array(test_scores_r)):.4f}")


In [ ]:
# Solution 9: Complete Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Create pipeline
numeric_features = ['age', 'fare', 'sibsp', 'parch']
categorical_features = ['pclass', 'sex', 'embarked']

numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = SimpleImputer(strategy='most_frequent')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Grid search on pipeline
param_grid_pipe = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10, None]
}

grid_pipe = GridSearchCV(pipeline, param_grid_pipe, cv=5, n_jobs=-1)
grid_pipe.fit(X_titanic, y_titanic)

print("Solution 9 - Pipeline Results:")
print(f"Best params: {grid_pipe.best_params_}")
print(f"Best CV score: {grid_pipe.best_score_:.4f}")


In [ ]:
# Solution 10: Heart Disease with Recall Optimization
from sklearn.datasets import fetch_openml
from sklearn.metrics import recall_score, precision_score, f1_score

# Load heart disease dataset
heart = fetch_openml(name='heart-statlog', version=1, parser='auto')
X_heart, y_heart = heart.data, heart.target
y_heart = (y_heart == 'present').astype(int)  # Convert to binary

X_train_hrt, X_test_hrt, y_train_hrt, y_test_hrt = train_test_split(
    X_heart, y_heart, test_size=0.2, random_state=42, stratify=y_heart)

# Optimize for recall (class_weight='balanced' helps with imbalanced data)
rf_heart = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',  # Important for medical diagnosis
    random_state=42
)

rf_heart.fit(X_train_hrt, y_train_hrt)
y_pred_hrt = rf_heart.predict(X_test_hrt)

print("Solution 10 - Heart Disease Detection:")
print(f"Recall (Sensitivity): {recall_score(y_test_hrt, y_pred_hrt):.4f}")
print(f"Precision: {precision_score(y_test_hrt, y_pred_hrt):.4f}")
print(f"F1-Score: {f1_score(y_test_hrt, y_pred_hrt):.4f}")
print(f"Accuracy: {accuracy_score(y_test_hrt, y_pred_hrt):.4f}")
print("\n💡 High recall ensures we catch most heart disease cases!")


---

## 🎉 Excellent!

Excellent! You have now mastered one of the most practical and powerful algorithms in machine learning. You can now:

- ✅ Understand ensemble learning and bagging
- ✅ Build and tune Random Forest models effectively
- ✅ Use OOB scores for validation
- ✅ Interpret feature importance reliably
- ✅ Control overfitting and optimize hyperparameters
- ✅ Apply Random Forest to both classification and regression

### What's Next? 🔮

**Tomorrow we explore XGBoost – The King of Gradient Boosting!**

We'll learn how sequential ensemble methods can push accuracy even higher, and why XGBoost dominates machine learning competitions.

---

*Python & AI Learning Path | Day 54 / 369* 🚀

**Keep growing your ML forest! 🌲🌳🌴**
